In [67]:
!kaggle datasets download tuannguyenvananh/iwslt15-englishvietnamese -p .

Dataset URL: https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamese
License(s): unknown
iwslt15-englishvietnamese.zip: Skipping, found more recently modified local copy (use --force to force download)


In [68]:
import zipfile
with zipfile.ZipFile("iwslt15-englishvietnamese.zip", "r") as zip_ref:
    zip_ref.extractall(".")  

In [69]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
from pyvi import ViTokenizer
import nltk
from nltk.tokenize import TreebankWordTokenizer
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [70]:
def load_data(en_path, vi_path):
    with open(en_path, encoding='utf-8') as f:
        en_lines = f.read().strip().split('\n')
    with open(vi_path, encoding='utf-8') as f:
        vi_lines = f.read().strip().split('\n')
    return en_lines, vi_lines

train_en, train_vi = load_data("iwslt'15 en-vi/train.en.txt", "iwslt'15 en-vi/train.vi.txt")
val_en, val_vi = load_data("iwslt'15 en-vi/tst2012.en.txt", "iwslt'15 en-vi/tst2012.vi.txt")
test_en, test_vi = load_data("iwslt'15 en-vi/tst2013.en.txt", "iwslt'15 en-vi/tst2013.vi.txt")


In [71]:
def tokenize(sentence, lang='en'):
    if lang=='en':
        return TreebankWordTokenizer().tokenize(sentence.lower())
    else: 
        return ViTokenizer.tokenize(sentence.lower())

def build_vocab(sentences, min_freq=2):
    freq = {}
    for sent in sentences:
        for word in tokenize(sent):
            freq[word] = freq.get(word, 0) + 1

    vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
    for word, count in freq.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

en_vocab = build_vocab(train_en)
vi_vocab = build_vocab(train_vi)
print(en_vocab)
print(vi_vocab)


{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'rachel': 4, 'pike': 5, ':': 6, 'the': 7, 'science': 8, 'behind': 9, 'a': 10, 'climate': 11, 'headline': 12, 'in': 13, '4': 14, 'minutes': 15, ',': 16, 'atmospheric': 17, 'chemist': 18, 'provides': 19, 'glimpse': 20, 'of': 21, 'massive': 22, 'scientific': 23, 'effort': 24, 'bold': 25, 'headlines': 26, 'on': 27, 'change': 28, 'with': 29, 'her': 30, 'team': 31, '--': 32, 'one': 33, 'thousands': 34, 'who': 35, 'contributed': 36, 'taking': 37, 'risky': 38, 'flight': 39, 'over': 40, 'rainforest': 41, 'pursuit': 42, 'data': 43, 'key': 44, 'molecule': 45, '.': 46, 'i': 47, '&': 48, 'apos': 49, ';': 50, 'd': 51, 'like': 52, 'to': 53, 'talk': 54, 'you': 55, 'today': 56, 'about': 57, 'scale': 58, 'that': 59, 'goes': 60, 'into': 61, 'making': 62, 'see': 63, 'paper': 64, 'look': 65, 'this': 66, 'when': 67, 'they': 68, 'have': 69, 'do': 70, 'and': 71, 'air': 72, 'quality': 73, 'or': 74, 'smog': 75, 'are': 76, 'both': 77, 'two': 78, 'branches': 79, 's

In [72]:
def encode(sentence, vocab):
    return [vocab.get(w, vocab['<unk>']) for w in tokenize(sentence)]

def pad_sequence(seq, max_len, pad_value=0):
    return seq[:max_len] + [pad_value] * (max_len - len(seq))


In [73]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.data = []
        for src, tgt in zip(src_sentences, tgt_sentences):
            src_ids = encode(src, src_vocab)
            tgt_ids = [tgt_vocab['<sos>']] + encode(tgt, tgt_vocab) + [tgt_vocab['<eos>']]
            if len(src_ids) < max_len and len(tgt_ids) < max_len:
                self.data.append((src_ids, tgt_ids))
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return torch.tensor(pad_sequence(src, self.max_len)), torch.tensor(pad_sequence(tgt, self.max_len))

    def __len__(self):
        return len(self.data)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    return torch.stack(src_batch), torch.stack(tgt_batch)

train_dataset = TranslationDataset(train_en, train_vi, en_vocab, vi_vocab)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)


In [74]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)  # combine both directions
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))  # [B, L, E]
        outputs, hidden = self.rnn(embedded)          # outputs: [B, L, 2H], hidden: [2, B, H]
        # Combine forward and backward hidden state
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden.unsqueeze(0)           # outputs: encoder outputs, hidden: decoder initial hidden


In [75]:
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hidden_dim] or [batch_size, 1, hidden_dim]
        # encoder_outputs: [batch_size, src_len, hidden_dim]

        if hidden.dim() == 2:
            hidden = hidden.unsqueeze(1)  # [B, 1, H]

        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.repeat(1, src_len, 1)  # [B, src_len, H]

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))  # [B, src_len, hidden_dim]
        attention = self.v(energy).squeeze(2)  # [B, src_len]
        return torch.softmax(attention, dim=1)




In [76]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, attention, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(enc_hid_dim * 2 + emb_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear(enc_hid_dim * 2 + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(1)  # [B, 1]
        embedded = self.dropout(self.embedding(input))  # [B, 1, E]

        attn_weights = self.attention(hidden.transpose(0, 1), encoder_outputs)  # [B, L]
        attn_weights = attn_weights.unsqueeze(1)  # [B, 1, L]

        context = torch.bmm(attn_weights, encoder_outputs)  # [B, 1, 2H]
        rnn_input = torch.cat((embedded, context), dim=2)   # [B, 1, E+2H]

        output, hidden = self.rnn(rnn_input, hidden)  # output: [B, 1, H], hidden: [1, B, H]

        output = output.squeeze(1)
        context = context.squeeze(1)
        embedded = embedded.squeeze(1)
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))  # [B, output_dim]

        return prediction, hidden


In [77]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        outputs = torch.zeros(B, T, self.decoder.output_dim).to(self.device)

        encoder_outputs, hidden = self.encoder(src)

        input = tgt[:, 0]
        for t in range(1, T):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1

        return outputs


In [78]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(vi_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
attn = Attention(HID_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, HID_DIM, attn)

model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


cuda


In [79]:
def train(model, iterator, optimizer, criterion, clip=1):
    model.train()
    epoch_loss = 0

    for src, tgt in tqdm(iterator, desc="Training", leave=False):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()

        output = model(src, tgt)  # output: [B, T, vocab_size]

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(iterator)


In [80]:
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt in tqdm(iterator, desc="Evaluating", leave=False):
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)  # No teacher forcing at eval

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


In [ ]:
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.3f}')


Epoch 1: Train Loss = 4.751


Epoch 2: Train Loss = 4.126


Epoch 3: Train Loss = 3.917


Epoch 4: Train Loss = 3.793


Training:  13%|█▎        | 242/1866 [05:21<35:55,  1.33s/it]

In [ ]:
def translate_sentence(model, sentence, src_vocab, tgt_vocab, max_len=50):
    model.eval()
    tokens = tokenize(sentence)
    src_ids = [src_vocab.get(tok, src_vocab['<unk>']) for tok in tokens]
    src_tensor = torch.tensor(pad_sequence(src_ids, max_len)).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    tgt_indices = [tgt_vocab['<sos>']]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_indices[-1]]).to(device)
        output, hidden = model.decoder(tgt_tensor, hidden, encoder_outputs)
        pred_token = output.argmax(1).item()
        if pred_token == tgt_vocab['<eos>']:
            break
        tgt_indices.append(pred_token)

    # Inverse vocab
    inv_vocab = {v: k for k, v in tgt_vocab.items()}
    translation = [inv_vocab.get(idx, '<unk>') for idx in tgt_indices[1:]]
    return translation


In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def calculate_bleu(model, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
    refs = []
    hyps = []

    for src, tgt in zip(src_sentences, tgt_sentences):
        prediction = translate_sentence(model, src, src_vocab, tgt_vocab)
        refs.append([tokenize(tgt)])  # Reference should be list of lists
        hyps.append(prediction)

    score = corpus_bleu(refs, hyps)
    return score

bleu_score = calculate_bleu(model, val_en, val_vi, en_vocab, vi_vocab)
print(f'BLEU score on validation set: {bleu_score:.4f}')


In [ ]:
def plot_attention(attention, sentence, translation):
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111)
    cax = ax.matshow(attention, cmap='bone')
    fig.colorbar(cax)

    ax.set_xticklabels([''] + tokenize(sentence), rotation=90)
    ax.set_yticklabels([''] + translation)

    plt.show()


In [ ]:
test_sentence = "How are you today?"
translation = translate_sentence(model, test_sentence, en_vocab, vi_vocab)
print(" ".join(translation))
